## Concept 5 — Conversational Agent

### What is it?
An agent that remembers previous turns. You maintain a `history` list of messages
and pass the full history to `create_agent` on every call. The agent sees all prior
context and can reference, correct, and build on earlier answers.

### vs Stateless Agent (Notebooks 1–4)
```
Stateless agent:
  Q1: 'What is 144/12?'  → A: '12'
  Q2: 'Multiply that by 5'  → A: 'I don't know what that refers to.'
  Every call starts fresh.

Conversational agent:
  Q1: 'What is 144/12?'  → A: '12'
  Q2: 'Multiply that by 5'  → A: '12 × 5 = 60'  ← remembers 12
  Q3: 'Summarize our session'  → full summary
```

### Pipeline
```
Turn 1: agent.invoke({'messages': [system, human_1]})            → ai_1
Turn 2: agent.invoke({'messages': [system, human_1, ai_1, human_2]}) → ai_2
Turn 3: agent.invoke({'messages': [system, ..., human_3]})       → ai_3
```

### Limitation (what Concept 6 fixes)
Tools are static — cannot dynamically retrieve from a real document store.
→ Concept 6 wraps a FAISS vectorstore as a tool, enabling real document retrieval.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

llm = get_llm()

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=(
        'You are a helpful conversational assistant. '
        'Use tools when needed. Reference prior turns when relevant.'
    ),
)
print('Conversational agent ready')

### Step 1 — Without Memory (Baseline)
Each call is a fresh invocation — the agent has no knowledge of earlier turns.

In [ ]:
# No history — two independent agent calls
r1 = agent.invoke({'messages': [{'role': 'user', 'content': 'What is 144 divided by 12?'}]})
print(f'Q1 answer: {r1["messages"][-1].content}')

r2 = agent.invoke({'messages': [{'role': 'user', 'content': 'Now multiply that result by 5.'}]})
print(f'Q2 answer: {r2["messages"][-1].content}')  # can't know what 'that result' is

### Step 2 — With Memory (Pass History Each Turn)
Maintain a `history` list. After each turn, extract the agent's final message
and append it so the next call sees the full conversation.

In [ ]:
history = []  # grows with each turn

def chat(user_input: str) -> str:
    history.append({'role': 'user', 'content': user_input})

    result     = agent.invoke({'messages': history})
    ai_message = result['messages'][-1]

    history.append({'role': 'assistant', 'content': ai_message.content})
    return ai_message.content

print('Chat with memory:')
print(f'Q1: {chat("What is 144 divided by 12?")}')
print(f'Q2: {chat("Now multiply that result by 5.")}')   # references previous answer
print(f'Q3: {chat("What is the weather in Hyderabad?")}')
print(f'Q4: {chat("Summarize what we just calculated and looked up.")}')

### Step 3 — Inspect History Growth
Each turn adds 2 messages (Human + AI). The agent always receives the full context.

In [ ]:
print(f'Total messages in history: {len(history)}')
for i, msg in enumerate(history, 1):
    role    = msg['role'].upper()
    content = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f'  [{i}] {role}: {content}')

### Step 4 — Window Memory Strategy
Full history grows unbounded. Window strategy: pass only the last N messages
to control token cost while keeping recent context.

In [ ]:
WINDOW = 4  # keep only last 4 messages (2 turns)
window_history = []

def chat_windowed(user_input: str) -> str:
    window_history.append({'role': 'user', 'content': user_input})

    # Use only the last WINDOW messages
    context    = window_history[-WINDOW:]
    result     = agent.invoke({'messages': context})
    ai_content = result['messages'][-1].content

    window_history.append({'role': 'assistant', 'content': ai_content})
    return ai_content

print('Windowed memory (last 4 messages only):')
print(f'A: {chat_windowed("What is 25 multiplied by 4?")}')
print(f'A: {chat_windowed("What is the weather in Delhi?")}')
print(f'A: {chat_windowed("What two things did we just look at?")}')
print(f'[Total stored: {len(window_history)}, passed to agent: {min(WINDOW, len(window_history))}]')

### Full Function

In [ ]:
session = []

def conversational_agent(query: str) -> str:
    session.append({'role': 'user', 'content': query})
    result  = agent.invoke({'messages': session})
    answer  = result['messages'][-1].content
    session.append({'role': 'assistant', 'content': answer})
    return answer

print('Session test:')
for q in TEST_QUERIES[:3]:
    print(f'\nQ: {q}')
    print(f'A: {conversational_agent(q)}')
    print(f'[Session length: {len(session)} messages]')